# 03_lstm_baseline

In [1]:
from datasets import load_from_disk
dataset = load_from_disk("../data/processed/iitb_filtered")

In [9]:
train_en = [ex["en"] for ex in dataset["train"]]
train_hi = [ex["hi"] for ex in dataset["train"]]

In [11]:
val_en = [ex["en"] for ex in dataset["validation"]]
val_hi = [ex["hi"] for ex in dataset["validation"]]

In [12]:
test_en = [ex["en"] for ex in dataset["test"]]
test_hi = [ex["hi"] for ex in dataset["test"]]

In [7]:
import sys
sys.path.append("../src")

In [8]:
import pickle
from vocab import Vocab

with open("../data/processed/en_vocab.pkl", "rb") as f:
    en_vocab = pickle.load(f)

with open("../data/processed/hi_vocab.pkl", "rb") as f:
    hi_vocab = pickle.load(f)

print(type(en_vocab), len(en_vocab))
print(type(hi_vocab), len(hi_vocab))

<class 'vocab.Vocab'> 114395
<class 'vocab.Vocab'> 105200


In [3]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [13]:
from dataset import TranslationDataset, collate_fn
from torch.utils.data import DataLoader
from functools import partial

train_dataset = TranslationDataset(train_en, train_hi, en_vocab, hi_vocab)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,
                     collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX))

src, tgt, src_lens, tgt_lens = next(iter(train_loader))
print(src.shape, tgt.shape)

torch.Size([32, 57]) torch.Size([32, 56])


In [18]:
val_dataset = TranslationDataset(val_en, val_hi, en_vocab, hi_vocab)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False,
                        collate_fn=partial(collate_fn, pad_idx=en_vocab.PAD_IDX))

In [14]:
from models.lstm_se12seq import Encoder, Decoder, Seq2Seq

EMBED_DIM = 256
HIDDEN_DIM = 512

encoder = Encoder(len(en_vocab), EMBED_DIM, HIDDEN_DIM, pad_idx=en_vocab.PAD_IDX).to(device)
decoder = Decoder(len(hi_vocab), EMBED_DIM, HIDDEN_DIM, pad_idx=hi_vocab.PAD_IDX).to(device)
model = Seq2Seq(encoder, decoder, device, start_idx=hi_vocab.START_IDX, end_idx=hi_vocab.END_IDX).to(device)

In [15]:
criterion = torch.nn.CrossEntropyLoss(ignore_index=hi_vocab.PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [19]:
from train import train_epoch, evaluate_epoch

NUM_EPOCHS = 5

for epoch in range(NUM_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion, device, teacher_forcing_ratio=0.5)
    val_loss = evaluate_epoch(model, val_loader, criterion, device)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} — Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f}")

AcceleratorError: CUDA error: out of memory
Search for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
The CUDA driver logged these messages, which may provide useful details:
Sticky error detected
Returning 2 (CUDA_ERROR_OUT_OF_MEMORY) from cuMemFree_v2

